## 1. Classes
| Class ID | Class Name |
|:--------:|------------|
| 0 | Curbs |
| 1 | Dents |
| 2 | Fence |
| 3 | Nature |
| 4 | Poles |
| 5 | Tallgrass |
| 6 | Vehicles |

```text
tripping_hazards/
??? dataset/
?   ??? images/
?   ?   ??? train/
?   ?   ??? val/
?   ??? labels/
?       ??? train/
?       ??? val/
??? runs/
```

Allow HEIC image acceptance but let model work on JPG version.

In [ ]:
%pip install -q ultralytics pillow-heif

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Configure dataset

In [ ]:
from pathlib import Path
import yaml
from ultralytics import YOLO
from PIL import Image
from IPython.display import display

CLASS_NAMES = [
    "curbs",
    "dents",
    "fence",
    "nature",
    "poles",
    "tallgrass",
    "vehicles",
]
CLASS_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}

PROJECT_ROOT = Path("/content/drive/MyDrive/tripping_hazards")
DATASET_ROOT = PROJECT_ROOT / "dataset"
DATA_YAML = PROJECT_ROOT / "data.yaml"
RUNS_DIR = PROJECT_ROOT / "runs"

DATA_CONFIG = {
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {index: name for index, name in enumerate(CLASS_NAMES)},
}
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_YAML.write_text(yaml.safe_dump(DATA_CONFIG, sort_keys=False), encoding="utf-8")
print(f"Dataset config written to: {DATA_YAML}")
print(f"Class mapping: {CLASS_TO_ID}")

## 3. Verify dataset

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def split_is_ready(split_name: str) -> tuple[bool, str]:
    image_dir = DATASET_ROOT / "images" / split_name
    label_dir = DATASET_ROOT / "labels" / split_name
    if not image_dir.is_dir() or not label_dir.is_dir():
        return False, f"Missing {split_name} image or label directory"

    images = [path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS]
    if not images:
        return False, f"No supported images found in {image_dir}"

    missing_labels = [image.stem for image in images if not (label_dir / f"{image.stem}.txt").is_file()]
    if missing_labels:
        sample = ", ".join(missing_labels[:5])
        return False, f"{len(missing_labels)} image(s) have no matching label file; examples: {sample}"

    return True, f"{len(images)} image(s) and matching labels found in {split_name}"

split_status = [split_is_ready(name) for name in ("train", "val")]
DATASET_READY = all(ready for ready, _ in split_status)
for _, message in split_status:
    print(message)
if not DATASET_READY:
    print("\nDataset not ready. Add the annotated dataset using the layout above; training and evaluation will be skipped.")

## 4. Segmentation model

In [ ]:
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 8
BEST_MODEL = RUNS_DIR / "tripping-hazards-seg" / "weights" / "best.pt"

if DATASET_READY:
    model = YOLO("yolov8s-seg.pt")
    train_results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        project=str(RUNS_DIR),
        name="tripping-hazards-seg",
    )
    BEST_MODEL = Path(train_results.save_dir) / "weights" / "best.pt"
    print(f"Best checkpoint: {BEST_MODEL}")
else:
    print("Training skipped: provide the annotated train/val segmentation dataset first.")

In [ ]:
if BEST_MODEL.is_file() and DATASET_READY:
    trained_model = YOLO(str(BEST_MODEL))
    metrics = trained_model.val(data=str(DATA_YAML), split="val")

    print(f"Mask precision: {metrics.seg.mp:.4f}")
    print(f"Mask recall:    {metrics.seg.mr:.4f}")
    print(f"Mask mAP50:     {metrics.seg.map50:.4f}")
    print(f"Mask mAP50-95:  {metrics.seg.map:.4f}")
    print(f"Box mAP50-95:   {metrics.box.map:.4f}")
    for class_name, class_map in zip(CLASS_NAMES, metrics.seg.maps):
        print(f"{class_name}: mask mAP50-95 = {class_map:.4f}")
else:
    print("Evaluation skipped: a trained best.pt checkpoint and ready dataset are required.")

In [ ]:
import tempfile
from pillow_heif import register_heif_opener

INPUT_IMAGE = Path("/content/drive/MyDrive/tripping_hazards/example.heic")


def predict_image(model_path: Path, image_path: Path):
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Input image not found: {image_path}")

    suffix = image_path.suffix.lower()
    model = YOLO(str(model_path))
    predict_options = {
        "imgsz": IMAGE_SIZE,
        "conf": 0.25,
        "save": True,
        "project": str(RUNS_DIR),
        "name": "predictions",
        "exist_ok": True,
    }

    if suffix in {".jpg", ".jpeg"}:
        return model.predict(source=str(image_path), **predict_options)
    if suffix in {".heic", ".heif"}:
        register_heif_opener()
        with Image.open(image_path) as image:
            rgb_image = image.convert("RGB")
        with tempfile.TemporaryDirectory(prefix="yolo_heic_") as temporary_dir:
            jpeg_path = Path(temporary_dir) / "converted.jpg"
            rgb_image.save(jpeg_path, format="JPEG", quality=95)
            return model.predict(source=str(jpeg_path), **predict_options)

    raise ValueError("Supported input formats are .jpg, .jpeg, .heic, and .heif")

if BEST_MODEL.is_file():
    prediction_results = predict_image(BEST_MODEL, INPUT_IMAGE)
    display(Image.fromarray(prediction_results[0].plot()[:, :, ::-1].copy()))
else:
    print("Prediction skipped: train the segmentation model and set INPUT_IMAGE first.")